# 05b — Exportação dos modelos ajustados para o firmware de inferência (v5)

Converte os modelos do **notebook 11** (CNN de 4 níveis acústicos, LSTM ajustado e camada de fusão) nos `.h` do firmware **v5**:

| Arquivo | Conteúdo |
|---|---|
| `modelo_cnn.h` | CNN int8 com **4 saídas** (níveis acústicos 0 a 3), densas por tensor |
| `parametros_modelos.h` | escalas e zeros da CNN, `CNN_CLASSES = 4`, faixas min-max do LSTM, arena mínima |
| `pesos_lstm.h` | pesos float32 do LSTM ajustado, para o `lstm_c.h` |
| `fusao.h` | **camada de fusão**: matriz 8 × 4 e 4 vieses (entrada: 4 prob. do LSTM + média das 4 prob. da CNN em 10 janelas) |
| `ops_modelos.h` | operações da CNN para o `MicroMutableOpResolver` |
| `tabelas_mel.h` | Hann, banco mel e tabelas da FFT (iguais às do notebook 05) |
| `vetores_teste.h` | paridade de espectrograma, CNN, LSTM e fusão, **só para o Teste_Inferencia v5** |

Também confere, em Python, que a conta da fusão e a do LSTM em C reproduzem os modelos Keras, e mede a arena da CNN no TFLite Micro.

**Antes:** rode o notebook 11 com o conjunto desejado. **Saída:** `dissertacao_tinyml/firmware_v5/` (e `firmware_v5_headers.zip`).

In [6]:
# 1. Ambiente e caminhos
import os, sys, re, json, zipfile, subprocess, importlib.util
if importlib.util.find_spec("tensorflow") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tensorflow==2.21.0"], check=True)
import numpy as np, pandas as pd, tensorflow as tf
try:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE = "/content/drive/MyDrive/dissertacao_tinyml"
except ImportError:
    BASE = os.environ.get("BASE_TINYML", "dissertacao_tinyml")
CONJUNTO = os.environ.get("CONJUNTO_TINYML", "coleta_sim")   # "coleta_sim" (sessões sintéticas) | "coleta" (dados do nó)
SIM = CONJUNTO != "coleta"
MOD = f"{BASE}/modelos/" + ("sim" if SIM else "insitu")
MIMII, BRISTOL = f"{BASE}/mimii_fan_6db", f"{BASE}/bristol_preparado"
SAIDA = f"{BASE}/firmware_v5"; os.makedirs(SAIDA, exist_ok=True)
print("Modelos de:", MOD)
try:
    from ai_edge_litert.interpreter import Interpreter
except ImportError:
    Interpreter = tf.lite.Interpreter
try:
    from tflite_micro.python.tflite_micro import runtime as tflm
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tflite-micro"], check=False)
    try:
        from tflite_micro.python.tflite_micro import runtime as tflm
    except ImportError:
        tflm = None
print("TFLite Micro no host:", "disponível" if tflm else "indisponível (a arena fica com o valor padrão)")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Modelos de: /content/drive/MyDrive/dissertacao_tinyml/modelos/sim
TFLite Micro no host: disponível


In [7]:
# 2. Modelos do notebook 11
q = json.load(open(f"{MOD}/cnn_insitu_quantizacao.json"))
cnn_b = open(f"{MOD}/cnn_insitu_int8.tflite", "rb").read()
MU, SD = np.array(q["linha_de_base"]["media"], np.float32), np.array(q["linha_de_base"]["desvio"], np.float32)
it = Interpreter(model_content=cnn_b); it.allocate_tensors()
E, S = it.get_input_details()[0], it.get_output_details()[0]
Q_ENT, Q_SAI = [float(v) for v in E["quantization"]], [float(v) for v in S["quantization"]]
N_CLASSES_CNN = int(S["shape"][-1])
assert list(E["shape"]) == [1, 98, 40, 1] and E["dtype"] == np.int8 and N_CLASSES_CNN == 4, (E["shape"], S["shape"])
por_canal = [t["name"] for t in it.get_tensor_details() if len(t["quantization_parameters"]["scales"]) > 1 and "MatMul" in t["name"]]
assert not por_canal, f"densas por canal (incompatíveis com a Harvard_TinyMLx): {por_canal}"
print(f"CNN int8: {len(cnn_b) / 1024:.1f} KB | saídas: {N_CLASSES_CNN} | entrada (escala, zero) {Q_ENT} | saída {Q_SAI}")

lstm = tf.keras.models.load_model(f"{MOD}/lstm_insitu.keras")
Wk, Wr, bl = lstm.get_layer("lstm").get_weights()
Wd, bd = lstm.get_layer("densa").get_weights(); Wo, bo = lstm.get_layer("saida").get_weights()
U = Wr.shape[0]; DENSA = Wd.shape[1]
fz = json.load(open(f"{MOD}/fusao.json")); W = np.array(fz["W"], np.float32); B = np.array(fz["b"], np.float32)
assert W.shape == (8, 4) and B.shape == (4,)
print(f"LSTM: {U} unidades, densa {DENSA} | fusão: W {W.shape}, b {B.shape}")

CNN int8: 12.6 KB | saídas: 4 | entrada (escala, zero) [0.05659332126379013, -27.0] | saída [0.00390625, -128.0]
LSTM: 32 unidades, densa 16 | fusão: W (8, 4), b (4,)


In [8]:
# 3. Conferências em Python: LSTM em C e fusão, contra os modelos Keras
FX = [tuple(v) for v in json.load(open(f"{BRISTOL}/params_bristol.json"))["faixas_minmax"].values()]
def atributos(v):
    v = np.asarray(v, np.float32); dp = v[..., 2] - v[..., 2].mean(axis=-1, keepdims=True)
    cols = [v[..., 0], v[..., 1], dp]
    return np.stack([np.clip((c - lo) / (hi - lo), 0, 1) for c, (lo, hi) in zip(cols, FX)], -1).astype(np.float32)
def lstm_c(Xn):                                  # a mesma conta do lstm_c.h, em float32
    sig = lambda v: 1 / (1 + np.exp(-v)); out = np.zeros((len(Xn), 4), np.float32)
    for n in range(len(Xn)):
        h = np.zeros(U, np.float32); c = np.zeros(U, np.float32)
        for t in range(Xn.shape[1]):
            z = Xn[n, t] @ Wk + h @ Wr + bl
            i, f, g, o = sig(z[:U]), sig(z[U:2 * U]), np.tanh(z[2 * U:3 * U]), sig(z[3 * U:])
            c = f * c + i * g; h = o * np.tanh(c)
        d = np.maximum(h @ Wd + bd, 0); e = np.exp(d @ Wo + bo - (d @ Wo + bo).max()); out[n] = e / e.sum()
    return out
bt = np.load(f"{BRISTOL}/teste.npz")
amostra = atributos(bt["X_bruto"][:50])
dif = np.abs(lstm_c(amostra) - lstm.predict(amostra, verbose=0)).max()
print(f"LSTM em C (NumPy) × Keras: diferença máxima {dif:.2e}"); assert dif < 1e-4
def fusao(pl, pa):                               # a mesma conta do firmware v5 (classePorFusao)
    zf = np.concatenate([pl, pa], -1) @ W + B; e = np.exp(zf - zf.max(-1, keepdims=True)); return e / e.sum(-1, keepdims=True)
fz_k = tf.keras.models.load_model(f"{MOD}/fusao.keras")
xr = np.random.default_rng(0).dirichlet(np.ones(4), size=(200, 2)).reshape(200, 8).astype(np.float32)
dif_f = np.abs(fusao(xr[:, :4], xr[:, 4:]) - fz_k.predict(xr, verbose=0)).max()
print(f"Fusão (conta do firmware) × Keras: diferença máxima {dif_f:.2e}"); assert dif_f < 1e-5

LSTM em C (NumPy) × Keras: diferença máxima 1.07e-06
Fusão (conta do firmware) × Keras: diferença máxima 2.98e-07


In [9]:
# 4. Operações e arena da CNN no TFLite Micro
def ops_do_modelo(b):
    i = Interpreter(model_content=b); i.allocate_tensors()
    return sorted({d["op_name"] for d in i._get_ops_details()} - {"DELEGATE"})
OPS = ops_do_modelo(cnn_b); print("Operações da CNN:", OPS)
ARENA = None
if tflm:
    def roda(arena):
        try:
            m = tflm.Interpreter.from_bytes(cnn_b, arena_size=arena)
            m.set_input(np.zeros((1, 98, 40, 1), np.int8), 0); m.invoke(); return True
        except Exception:
            return False
    lo, hi = 512, 512 * 1024
    while hi - lo > 64:
        mid = (lo + hi) // 2; lo, hi = (lo, mid) if roda(mid) else (mid, hi)
    ARENA = hi; print(f"Arena mínima da CNN no TFLM atual: {ARENA / 1024:.1f} KB")
    if ARENA > 14 * 1024:
        print(f"ATENÇÃO: o firmware v5 reserva 16 KB (ARENA_CNN_BYTES). Aumente para pelo menos {int(np.ceil(ARENA / 1024)) + 2} KB.")

Operações da CNN: ['CONV_2D', 'FULLY_CONNECTED', 'MAX_POOL_2D', 'MEAN', 'SOFTMAX']
Arena mínima da CNN no TFLM atual: 14.5 KB
ATENÇÃO: o firmware v5 reserva 16 KB (ARENA_CNN_BYTES). Aumente para pelo menos 17 KB.


In [10]:
# 5. Geração dos .h
def num(v, tipo):
    if tipo == "float":
        t = f"{float(v):.9g}"; return (t if any(ch in t for ch in ".en") else t + ".0") + "f"
    return str(int(v))
def vetor(tipo, nome, dados, por_linha=12, alinhado=False):
    dados = np.asarray(dados).ravel()
    linhas = [", ".join(num(v, tipo) for v in dados[i:i + por_linha]) for i in range(0, len(dados), por_linha)]
    return f"{'alignas(16) ' if alinhado else ''}const {tipo} {nome}[{len(dados)}] = {{\n  " + ",\n  ".join(linhas) + "\n};\n"
CAB = f"#pragma once\n// Gerado por 05b_exportar_firmware_v5.ipynb ({'sessões sintéticas' if SIM else 'dados do nó'}) — não editar\n#include <stdint.h>\n\n"
A = {}
A["modelo_cnn.h"] = CAB + vetor("unsigned char", "g_modelo_cnn", np.frombuffer(cnn_b, np.uint8), 16, True) + f"const unsigned int g_modelo_cnn_len = {len(cnn_b)};\n"
A["pesos_lstm.h"] = (CAB + "// Pesos do LSTM ajustado (notebook 11) para o lstm_c.h. Matrizes [entrada][saída]; portas i, f, g, o.\n"
    + vetor("float", "LSTM_WK", Wk, 8) + vetor("float", "LSTM_WR", Wr, 8) + vetor("float", "LSTM_B", bl, 8)
    + vetor("float", "LSTM_WD", Wd, 8) + vetor("float", "LSTM_BD", bd, 8) + vetor("float", "LSTM_WO", Wo, 8) + vetor("float", "LSTM_BO", bo, 4))
A["fusao.h"] = (CAB + "// Camada de fusão treinada (notebook 11): z = x W + b; classe = argmax(softmax(z)).\n"
    + "// x = [p_LSTM(Normal, Atenção, Crítico, Falha), média de 10 janelas de p_CNN(nível 0, 1, 2, 3)]; W[i][j] = FUSAO_W[i * 4 + j].\n"
    + "const int FUSAO_ENTRADAS = 8, FUSAO_CLASSES = 4;\n" + vetor("float", "FUSAO_W", W, 4) + vetor("float", "FUSAO_B", B, 4))
METODO = {"CONV_2D": "AddConv2D", "MAX_POOL_2D": "AddMaxPool2D", "AVERAGE_POOL_2D": "AddAveragePool2D", "MEAN": "AddMean",
          "FULLY_CONNECTED": "AddFullyConnected", "SOFTMAX": "AddSoftmax", "RESHAPE": "AddReshape", "QUANTIZE": "AddQuantize",
          "DEQUANTIZE": "AddDequantize", "ADD": "AddAdd", "MUL": "AddMul"}
assert all(o in METODO for o in OPS), [o for o in OPS if o not in METODO]
A["ops_modelos.h"] = (CAB + f"// Operações usadas pela CNN: {', '.join(OPS)} (o LSTM roda em C, sem interpretador)\n"
    f"#define N_OPS_MODELOS {len(OPS)}\n\ntemplate <typename R>\nbool registrarOps(R& r) {{\n  return\n"
    + " &&\n".join(f"    r.{METODO[o]}() == kTfLiteOk" for o in OPS) + ";\n}\n")
A["parametros_modelos.h"] = CAB + f'''// CNN 2D int8 ajustada (notebook 11): entrada 98 x 40 x 1, saída = 4 níveis acústicos (0 a 3)
const int   CNN_QUADROS = 98, CNN_FILTROS_MEL = 40, CNN_CLASSES = {N_CLASSES_CNN};
const float CNN_ESCALA_ENTRADA = {num(Q_ENT[0], "float")};
const int   CNN_ZERO_ENTRADA   = {int(Q_ENT[1])};
const float CNN_ESCALA_SAIDA   = {num(Q_SAI[0], "float")};
const int   CNN_ZERO_SAIDA     = {int(Q_SAI[1])};

// LSTM (float32, executado em C): 30 passos de 10 s, 3 atributos, {U} unidades, 4 classes
const int LSTM_PASSOS = 30, LSTM_ATRIBUTOS = 3, LSTM_UNIDADES = {U}, LSTM_CLASSES = 4;
const int LSTM_ENT_X = 0, LSTM_ENT_H = 1, LSTM_ENT_C = 2, LSTM_SAI_H = 0, LSTM_SAI_C = 1, LSTM_SAI_P = 2;   // só com USAR_LSTM_C = 0

// Normalização min-max dos atributos (notebook 02): temperatura, umidade, pressão - média da janela
const float FAIXA_TEMP[2] = {{{num(FX[0][0], "float")}, {num(FX[0][1], "float")}}};
const float FAIXA_UMID[2] = {{{num(FX[1][0], "float")}, {num(FX[1][1], "float")}}};
const float FAIXA_DPRESS[2] = {{{num(FX[2][0], "float")}, {num(FX[2][1], "float")}}};

const unsigned int ARENA_MINIMA_CNN = {ARENA or 16 * 1024}, ARENA_MINIMA_LSTM = 0;
'''
hann = np.load(f"{MIMII}/hann.npy").astype(np.float32); banco = np.load(f"{MIMII}/banco_mel.npy").astype(np.float32)
ini, tam, desl, pesos = [], [], [], []
for linha in banco:
    nz = np.nonzero(linha)[0]; a, b = (int(nz[0]), int(nz[-1]) + 1) if len(nz) else (0, 0)
    ini.append(a); tam.append(b - a); desl.append(len(pesos)); pesos.extend(linha[a:b])
k = np.arange(128); kk = np.arange(256); rev = np.array([int(f"{i:08b}"[::-1], 2) for i in range(256)])
A["tabelas_mel.h"] = (CAB + vetor("float", "HANN", hann, 8) + vetor("uint16_t", "MEL_INICIO", ini, 20) + vetor("uint16_t", "MEL_TAM", tam, 20)
    + vetor("uint16_t", "MEL_DESLOC", desl, 20) + vetor("float", "MEL_PESOS", pesos, 8)
    + vetor("float", "FFT_COS", np.cos(2 * np.pi * k / 256), 8) + vetor("float", "FFT_SIN", -np.sin(2 * np.pi * k / 256), 8)
    + vetor("float", "RFFT_COS", np.cos(2 * np.pi * kk / 512), 8) + vetor("float", "RFFT_SIN", -np.sin(2 * np.pi * kk / 512), 8)
    + vetor("uint16_t", "FFT_REV", rev, 16))

# Vetores de teste: 2 janelas de áudio (paridade), normalizadas pela linha de base usada no treino; LSTM; fusão
par = np.load(f"{MIMII}/paridade.npz", allow_pickle=True)
sel = [int(np.where(par["rotulo"] == r)[0][0]) for r in (0, 1)]
lm = par["logmel"][sel].astype(np.float32)
qz = np.clip(np.round(((lm - MU) / SD) / Q_ENT[0] + Q_ENT[1]), -128, 127).astype(np.int8)
saida_q = []
for i in range(2):
    it.set_tensor(E["index"], qz[i][None, ..., None]); it.invoke(); saida_q.append(it.get_tensor(S["index"])[0])
p_cnn = (np.array(saida_q, np.float32) - Q_SAI[1]) * Q_SAI[0]
real = np.where(bt["sintetica"] == 0)[0] if "sintetica" in bt.files else np.arange(len(bt["y"]))
sel_l = [int(real[np.where(bt["y"][real] == c)[0][0]]) for c in range(4) if (bt["y"][real] == c).any()]
bruto = bt["X_bruto"][sel_l].astype(np.float32); xn = atributos(bruto); p_l = lstm_c(xn)
f_ent = np.concatenate([p_l[: len(p_cnn)], p_cnn[: len(p_l)]], 1)       # pares (LSTM k, CNN k) para conferir a fusão
f_ent = np.vstack([f_ent, np.random.default_rng(1).dirichlet(np.ones(4), size=(6, 2)).reshape(6, 8)]).astype(np.float32)   # + 6 casos sorteados
f_prob = fusao(f_ent[:, :4], f_ent[:, 4:])
A["vetores_teste.h"] = (CAB + "const int N_TESTE_AUDIO = 2;\n" + vetor("int16_t", "TESTE_AUDIO", par["audio_int16"][sel], 16)
    + vetor("float", "TESTE_LOGMEL", lm, 8) + vetor("float", "TESTE_BASE_MEDIA", np.tile(MU, 2), 8) + vetor("float", "TESTE_BASE_DESVIO", np.tile(SD, 2), 8)
    + vetor("int8_t", "TESTE_CNN_ENTRADA", qz, 20) + vetor("int8_t", "TESTE_CNN_SAIDA", np.array(saida_q), 4)
    + f"const int N_TESTE_LSTM = {len(sel_l)};\n" + vetor("int8_t", "TESTE_LSTM_CLASSE", bt["y"][sel_l], 4)
    + vetor("float", "TESTE_LSTM_BRUTO", bruto, 9) + vetor("float", "TESTE_LSTM_ATRIB", xn, 9) + vetor("float", "TESTE_LSTM_PROB", p_l, 4)
    + f"const int N_TESTE_FUSAO = {len(f_ent)};\n" + vetor("float", "TESTE_FUSAO_ENT", f_ent, 8) + vetor("float", "TESTE_FUSAO_PROB", f_prob, 4))
with zipfile.ZipFile(f"{SAIDA}/firmware_v5_headers.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    for nome, txt in A.items():
        open(f"{SAIDA}/{nome}", "w", encoding="utf-8").write(txt); zf.writestr(nome, txt)
for nome in A: print(f"{nome:22s} {os.path.getsize(f'{SAIDA}/{nome}') / 1024:7.1f} KB (texto)")
print("CNN esperada nas janelas de teste (int8):", [list(map(int, s)) for s in saida_q])

modelo_cnn.h              53.2 KB (texto)
pesos_lstm.h              74.8 KB (texto)
fusao.h                    0.9 KB (texto)
ops_modelos.h              0.5 KB (texto)
parametros_modelos.h       1.1 KB (texto)
tabelas_mel.h             25.7 KB (texto)
vetores_teste.h          323.6 KB (texto)
CNN esperada nas janelas de teste (int8): [[32, -52, -107, -128], [89, -112, -105, -128]]


## Próximos passos

1. Copie os `.h` de `firmware_v5/` (ou o zip) para as duas pastas:
   * `Teste_Inferencia_v5/`: **todos** os `.h`, mais `espectro_mel.h`, `preprocessamento.h`, `inferencia.h` e `lstm_c.h` do pacote v5;
   * `Multimodais-de-Monitorizacao_v5/`: todos, **exceto** `vetores_teste.h`, mais o `secrets.h`.
2. Grave o **Teste_Inferencia_v5**: espectrograma, CNN, LSTM e fusão devem dar **OK**, e a decisão no pior caso deve ficar abaixo de 200 ms (refaz a Tabela 12).
3. Grave o **firmware v5** e confira no Monitor Serial a linha `>> Fusão: camada densa (8 -> 4)`. Na compilação, anote **Flash** e **RAM** (margem de 1,3 KB na Flash em relação ao RNF3).